### Лабораторная работа №8: Организация запросов для обращения к базе данных на естественном языке

In [ ]:
!pip install transformers==5.7.0
!pip install torch-scatter -f https://data.pyg.org/whl/torch-1.9.0+${CUDA}.html

### Подготовка данных

In [2]:
from google.colab import drive
import pandas as pd

# Монтирование Google Диска
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Загрузка данных из CSV файла
data = pd.read_csv('/content/drive/MyDrive/MMoDM/Subscribers_202604301215.csv')

# Вывод таблицы на экран
data

,sub_ID,sub_name,sub_phone,sub_inn,sub_account
0,1,"ООО ""Альфа""",8-495-123-45-67,7712345678,40702810123450000001
1,2,"ЗАО ""Бета-Торг""",8-812-987-65-43,7898765432,40702810123450000002
2,3,"ПАО ""Гамма Строй""",8-843-111-22-33,1655123456,40702810123450000003
3,4,ИП Иванов А.А.,8-383-555-44-33,5401123456,40702810123450000004
4,5,"ООО ""Дельта-IT""",8-343-777-88-99,6671123456,40702810123450000005
5,6,"АО ""Омега""",8-831-222-33-44,5260123456,40702810123450000006
6,7,"ООО ""Сигма""",8-351-444-55-66,7453123456,40702810123450000007
7,8,ИП Петров В.С.,8-846-666-77-88,6316123456,40702810123450000008
8,9,"ЗАО ""Вектор""",8-381-888-99-00,5503123456,40702810123450000009
9,10,"ПАО ""Монолит""",8-863-101-01-01,6164123456,40702810123450000010


### Инициализация нейросетевой модели

In [4]:
from transformers import AutoModelForTableQuestionAnswering, AutoTokenizer

# Конвертация датафрейма в строковый тип
data = data.astype(str)

# Загрузка модели и токенизатора с принудительным включением необходимых полей
model_name = 'google/tapas-base-finetuned-wtq'
tapas_tokenizer = AutoTokenizer.from_pretrained(model_name)
tapas_model = AutoModelForTableQuestionAnswering.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

### Настройка Pipeline и функции запросов

In [ ]:
from transformers import pipeline

# Инициализация конвейера (pipeline) для задачи table-question-answering
nlp = pipeline('table-question-answering', model=tapas_model, tokenizer=tapas_tokenizer)

# Функция для выполнения запросов и форматированного вывода результата
def qa(query, data):
    print('>>>>>')
    print(query)
    # Передача таблицы и запроса в нейросетевую модель
    result = nlp({'table': data, 'query': query})

    # Извлечение только найденных ячеек с ответом
    answer = result['cells']
    print(answer)

### Выполнение запросов на естественном языке

In [9]:
print(">>>>> Запрос 1: Узнать ИНН по ID абонента")
qa('What is the sub_inn of sub_ID 4?', data)

print("\n>>>>> Запрос 2: Узнать номер телефона")
qa('What is the sub_phone for ООО "Спектр"?', data)

print("\n>>>>> Запрос 3: Узнать название компании по её ID")
qa('What is the sub_name of sub_ID 15?', data)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


>>>>> Запрос 1: Узнать ИНН по ID абонента
>>>>>
What is the sub_inn of sub_ID 4?
['5401123456']

>>>>> Запрос 2: Узнать номер телефона
>>>>>
What is the sub_phone for ООО "Спектр"?
['8-347-202-02-02']

>>>>> Запрос 3: Узнать название компании по её ID
>>>>>
What is the sub_name of sub_ID 15?
['ЗАО "Авангард"']
